<a href="https://colab.research.google.com/github/GunjanBhandari-pixel/Gunnu/blob/main/Delay_Example_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from matplotlib.ticker import ScalarFormatter
from mpmath import mp, matrix, zeros

mp.dps =50
a = mp.mpf("0.30")
g2 = mp.gamma(4-a)
gamma_const = 1 / mp.gamma(2-a)
g = mp.gamma(3-a)
an=1
one=mp.mpf(1)
two=mp.mpf(2)
three=mp.mpf(3)

def f(a3,b3):
    return -1*b3*mp.sin(a3)

a_=1
c_=-1
L=mp.pi
T=1
M=128*2
#N=math.floor(T*((M/L)**(6/(3-a))))
N=8*M
h=L/M
#r=1
r=3/a-1

t = [mp.mpf('0') for _ in range(N+1)]

# First half: graded in [0,1]
for i in range(0, int(N/2)+1):
    t[i] = mp.power(i / int(N/2), r)

# Second half: shifted copy (required for delay consistency)
for i in range(int(N/2)+1, N+1):
    t[i] = 1 + t[i - int(N/2)]

Tau= mp.zeros(N+1, 1)

for i in range(1,N+1):
    Tau[i,0]=t[i]-t[i-1]
Tau[0,0]=Tau[1,0]

x = [mp.mpf(i)*L/M for i in range(M+1)]

X = [mp.mpf(0)]*(M+3)
X[0] = x[0]
X[1] = (x[0] + x[1]) / 2
for j in range(2, M+1):
    X[j] = x[j-1]
X[M+1] = (x[M] + x[M-1]) / 2
X[M+2] = x[M]

U=mp.zeros(M+3,N+1)
exact=mp.zeros(M+3,N+1)
error=mp.zeros(M+3,N+1)

A1= mp.zeros(M+5, M+5)
B=mp.zeros(M+5,M+5)
C=mp.zeros(N+1,M+5)
W1=mp.zeros(M+5,1)
W=mp.zeros(M+5,1)

c1=69120*(h**2)*(1/a_)
d1=4320*(h**2)*(1/a_)

A1[0,0]=1/120       # first boundary condition
A1[0,1]=26/120
A1[0,2]=66/120
A1[0,3]=26/120
A1[0,4]=1/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-717/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-1448/d1
A1[1,2]=4300/d1
A1[1,3]=-1322/d1
A1[1,4]=-938/d1
A1[1,5]=202/d1
A1[1,6]=-92/d1
A1[1,7]=10/d1
A1[1,8]=7/d1
A1[1,9]=-2/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-1475/c1         # Tau1
A1[2,1]=-30149/c1
A1[2,2]=31918/c1
A1[2,3]=30322/c1
A1[2,4]=-27776/c1
A1[2,5]=-3680/c1
A1[2,6]=994/c1
A1[2,7]=-98/c1
A1[2,8]=-77/c1
A1[2,9]=21/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=2/d1                                # y-y"(1)
A1[3,1]=-725/d1
A1[3,2]=-1454/d1
A1[3,3]=4396/d1
A1[3,4]=-1574/d1
A1[3,5]=-602/d1
A1[3,6]=-50/d1
A1[3,7]=4/d1
A1[3,8]=4/d1
A1[3,9]=-1/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=1/d1
    A1[i+2,-3+i+2]=-2/d1
    A1[i+2,-2+i+2]=-728/d1
    A1[i+2,-1+i+2]=-1406/d1
    A1[i+2,i+2]=4270/d1
    A1[i+2,1+i+2]=-1406/d1
    A1[i+2,2+i+2]=-728/d1
    A1[i+2,3+i+2]=-2/d1
    A1[i+2,4+i+2]=1/d1

def gamma_mat(k,j):
    return (mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a))*gamma_const

def Del(k,j):
    A5 = mp.power(t[j]-t[k], 2-a)
    B5 = mp.power(t[j]-t[k+1], 1-a)
    C5 = t[j] + (a-2)*t[k] + (1-a)*t[k+1]
    return (A5 - B5*C5)/g - ((Tau[k+1,0]-Tau[k,0])/2)*gamma_mat(k,j)

def dg(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / (Tau[j,0] * (Tau[j - 1,0] + Tau[j,0]))
            + gamma_mat(j - 1, j) / (Tau[j - 1,0] + Tau[j,0])
        )
    else:
        return 0

for i in range(0,M+3):
    U[i,0]=0

B=A1.copy()

for n in range(1,1+1):
    q1=(dg(1,1)-c_)/120
    q2=(dg(1,1)-c_)/3840

    B[1,0]=A1[1,0]+q1
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1


    num1=two*(mp.power(Tau[0,0],2))
    num2=two*Tau[0,0]
    W1[0,0]=0
    for i in range(0,M+3):
        s=(2*Del(0, 1)/num1-gamma_mat(0, 1)/num2)*f(X[i],-t[1])
        W1[i+1,0]=(one-t[n])*f(X[i],t[n]-1)+three*mp.sin(X[i])-s
    W1[M+4,0]=0

    G1=mp.lu_solve(B,W1)

    for i in range(0,M+5):
        C[n,i]=G1[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-1]+C[n,i]*26+C[n,i+1]*66+C[n,i+2]*26+C[n,i+3])/120

    U[M+1,n]=(C[n,M-1]+237*C[n,M]+1682*C[n,M+1]+1682*C[n,M+2]+237*C[n,M+3]+C[n,M+4])/3840
    U[M+2,n]=(C[n,M]+C[n,M+1]*26+C[n,M+2]*66+C[n,M+3]*26+C[n,M+4])/120


B=A1.copy()

for n in range(2,2+1):
    q1=(dg(2,2)-c_)/120
    q2=(dg(2,2)-c_)/3840

    B[1,0]=A1[1,0]+q1
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2

        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1

    W[0,0]=0
    for i in range(0,M+3):
        s=2*Del(0,2)*(U[i,1]/(Tau[1,0]*(Tau[1,0]+Tau[0,0]))+f(X[i],-t[1])/(Tau[0,0]*(Tau[1,0]+Tau[0,0])))\
        +gamma_mat(0,2)*((U[i,1]-f(X[i],-t[1]))/(Tau[1,0]+Tau[0,0]))
        W[i+1,0]=(1-t[n])*f(X[i],t[n]-1)+three*mp.sin(X[i])-s+U[i,1]*two*Del(1, 2)/(Tau[1,0]*Tau[2,0])
    W[M+4,0]=0

    G=  mp.lu_solve(B,W)

    for i in range(0,M+5):
        C[n,i]=G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-1]+C[n,i]*26+C[n,i+1]*66+C[n,i+2]*26+C[n,i+3])/120

    U[M+1,n]=(C[n,M-1]+237*C[n,M]+1682*C[n,M+1]+1682*C[n,M+2]+237*C[n,M+3]+C[n,M+4])/3840
    U[M+2,n]=(C[n,M]+C[n,M+1]*26+C[n,M+2]*66+C[n,M+3]*26+C[n,M+4])/120

B=A1.copy()
for n in range(3,int(N/2)+1):
    # --- D1 matrix ---
    D1 = mp.matrix(N+1,1)
    j=n
    D1[0,0] = -2 * Del(1, j) / (Tau[1,0] * (Tau[1,0] + Tau[2,0])) + gamma_mat(1, j) / (Tau[1,0] + Tau[2,0])
    D1[1,0] = (
        -2 * Del(2, j) / (Tau[2,0] * (Tau[2,0] + Tau[3,0]))
        + 2 * Del(1, j) / (Tau[1,0] * Tau[2,0])
        + gamma_mat(2, j) / (Tau[2,0] + Tau[3,0])
    )
    for k in range(2, j - 1):
        D1[k,0] = (
            -2 * Del(k - 1, j) / (Tau[k,0] * (Tau[k - 1,0] + Tau[k,0]))
            + 2 * Del(k, j) / (Tau[k,0] * Tau[k + 1,0])
            - 2 * Del(k + 1, j) / (Tau[k + 1,0] * (Tau[k + 1,0] + Tau[k + 2,0]))
            - gamma_mat(k - 1, j) / (Tau[k - 1,0] + Tau[k,0])
            + gamma_mat(k + 1, j) / (Tau[k + 1,0] + Tau[k + 2,0])
        )
    D1[j - 1,0] = (
        -2 * Del(j - 2, j) / (Tau[j - 1,0] * (Tau[j - 2,0] + Tau[j - 1,0]))
        + 2 * Del(j - 1, j) / (Tau[j - 1,0] * Tau[j,0])
        - gamma_mat(j - 2, j) / (Tau[j - 2,0] + Tau[j - 1,0])
    )
    D1[j,0] = (
        2 * Del(j - 1, j) / (Tau[j,0] * (Tau[j - 1,0] + Tau[j,0]))
        + gamma_mat(j - 1, j) / (Tau[j - 1,0] + Tau[j,0])
    )


    q1=(dg(n,n)-c_)/120
    q2=(dg(n,n)-c_)/3840

    B[1,0]=A1[1,0]+q1
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1


    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1

    W[0,0]=0
    for i in range(0,M+3):
        s=2*Del(0,n)*(U[i,1]/(Tau[1,0]*(Tau[1,0]+Tau[0,0]))+f(X[i],-t[1])/(Tau[0,0]*(Tau[1,0]+Tau[0,0])))+gamma_mat(0, n)*((U[i,1]-f(X[i],-t[1]))/(Tau[1,0]+Tau[0,0]))
        W[i+1,0]=mp.fdot(D1[0:n,0],U[i,0:n])-s+(1-t[n])*f(X[i],t[n]-1)+three*mp.sin(X[i])
    W[M+4,0]=0

    G=  mp.lu_solve(B,W)

    for i in range(0,M+5):
        C[n,i]=G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-1]+C[n,i]*26+C[n,i+1]*66+C[n,i+2]*26+C[n,i+3])/120

    U[M+1,n]=(C[n,M-1]+237*C[n,M]+1682*C[n,M+1]+1682*C[n,M+2]+237*C[n,M+3]+C[n,M+4])/3840
    U[M+2,n]=(C[n,M]+C[n,M+1]*26+C[n,M+2]*66+C[n,M+3]*26+C[n,M+4])/120

B=A1.copy()
for n in range(int(N/2)+1,N+1):
    # --- D1 matrix ---
    D1 = mp.matrix(N+1,1)
    j=n
    D1[0,0] = -2 * Del(1, j) / (Tau[1,0] * (Tau[1,0] + Tau[2,0])) + gamma_mat(1, j) / (Tau[1,0] + Tau[2,0])
    D1[1,0] = (
        -2 * Del(2, j) / (Tau[2,0] * (Tau[2,0] + Tau[3,0]))
        + 2 * Del(1, j) / (Tau[1,0] * Tau[2,0])
        + gamma_mat(2, j) / (Tau[2,0] + Tau[3,0])
    )
    for k in range(2, j - 1):
        D1[k,0] = (
            -2 * Del(k - 1, j) / (Tau[k,0] * (Tau[k - 1,0] + Tau[k,0]))
            + 2 * Del(k, j) / (Tau[k,0] * Tau[k + 1,0])
            - 2 * Del(k + 1, j) / (Tau[k + 1,0] * (Tau[k + 1,0] + Tau[k + 2,0]))
            - gamma_mat(k - 1, j) / (Tau[k - 1,0] + Tau[k,0])
            + gamma_mat(k + 1, j) / (Tau[k + 1,0] + Tau[k + 2,0])
        )
    D1[j - 1,0] = (
        -2 * Del(j - 2, j) / (Tau[j - 1,0] * (Tau[j - 2,0] + Tau[j - 1,0]))
        + 2 * Del(j - 1, j) / (Tau[j - 1,0] * Tau[j,0])
        - gamma_mat(j - 2, j) /(Tau[j - 2,0] + Tau[j - 1,0])
    )
    D1[j,0] = (
        2 * Del(j - 1, j) / (Tau[j,0] * (Tau[j - 1,0] + Tau[j,0]))
        + gamma_mat(j - 1, j) / (Tau[j - 1,0] + Tau[j,0])
    )

    q1=(dg(n,n)-c_)/120
    q2=(dg(n,n)-c_)/3840

    B[1,0]=A1[1,0]+q1
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1

    W[0,0]=0
    for i in range(0,M+3):
        s=2*Del(0,n)*(U[i,1]/(Tau[1,0]*(Tau[1,0]+Tau[0,0]))+f(X[i],-t[1])/(Tau[0,0]*(Tau[1,0]+Tau[0,0])))\
            +gamma_mat(0, n)*((U[i,1]-f(X[i],-t[1]))/(Tau[1,0]+Tau[0,0]))
        W[i+1,0]=mp.fdot(D1[0:n,0],U[i,0:n])+(t[n]-one)*U[i,n-int(N/2)]+3*t[n]*mp.sin(X[i])-s
    W[M+4,0]=0

    G=  mp.lu_solve(B,W)

    for i in range(0,M+5):
        C[n,i]=G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-1]+C[n,i]*26+C[n,i+1]*66+C[n,i+2]*26+C[n,i+3])/120

    U[M+1,n]=(C[n,M-1]+237*C[n,M]+1682*C[n,M+1]+1682*C[n,M+2]+237*C[n,M+3]+C[n,M+4])/3840
    U[M+2,n]=(C[n,M]+C[n,M+1]*26+C[n,M+2]*66+C[n,M+3]*26+C[n,M+4])/120

# U1 = np.array([[float(U[i, j]) for j in range(U.cols)]
#                               for i in range(U.rows)])

# np.savetxt("Example_1_node8.txt", U1, fmt="%.12e")

# U2 = np.array([[float(U[i, j]) for j in range(U.cols)]
#                               for i in range(U.rows)])

# np.savetxt("Example_1_node16.txt", U2, fmt="%.12e")

# U3 = np.array([[float(U[i, j]) for j in range(U.cols)]
#                               for i in range(U.rows)])

# np.savetxt("Example_1_node32.txt", U3, fmt="%.12e")

# U4 = np.array([[float(U[i, j]) for j in range(U.cols)]
#                               for i in range(U.rows)])

# np.savetxt("Example_1_node64.txt", U4, fmt="%.12e")

# U5= np.array([[float(U[i, j]) for j in range(U.cols)]
#                               for i in range(U.rows)])

# np.savetxt("Example_1_node128.txt", U5, fmt="%.12e")

U6= np.array([[float(U[i, j]) for j in range(U.cols)]
                              for i in range(U.rows)])

np.savetxt("Example_1_node256.txt", U6, fmt="%.12e")

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# =========================
# Meshgrid
# =========================
Xplot, Yplot = np.meshgrid(X, t)

Xplot = Xplot.astype(float)
Yplot = Yplot.astype(float)

# =========================
# Convert mpmath → NumPy (SAFE)
# U is (space, time) = (M+3, N+1)
# =========================
V_float = np.array([
    [float(U[i, j]) for j in range(N+1)]
    for i in range(M+3)
])

# =========================
# Fix orientation → (time, space)
# =========================
V_float = V_float.T

# =========================
# Safety check (VERY IMPORTANT)
# =========================
print("Xplot shape:", Xplot.shape)
print("V_float shape:", V_float.shape)

if V_float.shape != Xplot.shape:
    raise ValueError("Shape mismatch: check U construction!")

# =========================
# Plot settings
# =========================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120
})

# =========================
# Create figure
# =========================
fig = plt.figure(figsize=(7.5, 5.5))
ax = fig.add_subplot(111, projection='3d')

# =========================
# Surface plot
# =========================
surf = ax.plot_surface(
    Xplot, Yplot, V_float,
    cmap='viridis',
    linewidth=0,
    antialiased=False,
    rstride=1,
    cstride=1
)

surf.set_rasterized(True)

# =========================
# Labels
# =========================
ax.set_xlabel(r"Space $x$", labelpad=10)
ax.set_ylabel(r"Time $t$", labelpad=10)
ax.set_zlabel(r"Numerical solution", labelpad=10)

# =========================
# Scientific Z-axis
# =========================
formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))
ax.zaxis.set_major_formatter(formatter)

# =========================
# View and style
# =========================
ax.view_init(elev=28, azim=225)
ax.set_box_aspect([1, 1, 0.6])
ax.grid(False)

# =========================
# Colorbar (important)
# =========================
cbar = fig.colorbar(surf, shrink=0.6, aspect=12)
cbar.set_label("Solution magnitude")

# =========================
# Save figures
# =========================
plt.tight_layout()
plt.savefig("Numerical_solution.png", dpi=300, bbox_inches="tight")
plt.savefig("Numerical_solution.pdf", bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
from mpmath import mp
import matplotlib.pyplot as plt

U1 = np.loadtxt('Example_1_node32.txt')
U2 = np.loadtxt('Example_1_node64.txt')
three = mp.mpf(3)
two=mp.mpf(2)
one=mp.mpf(1)
M =32
N = 8*M

U_error = np.zeros((M+3, N+1))

for i in range(2, M+1):
    for j in range(1, N+1):
        U_error[i,j] = abs(U2[2*i-1,2*j] - U1[i,j])

print(np.max(U_error))

a = mp.mpf("0.30")
r = three/a -one
#r = two/a
#r=1
L=mp.pi

x = [mp.mpf(i)*L/M for i in range(M+1)]
X = [mp.mpf(0)]*(M+3)
X[0] = x[0]
X[1] = (x[0] + x[1]) / 2
for j in range(2, M+1):
    X[j] = x[j-1]
X[M+1] = (x[M] + x[M-1]) / 2
X[M+2] = x[M]

half = N // 2  # safer and cleaner

t = [mp.mpf('0') for _ in range(N+1)]

# First half: graded mesh on [0,1]
for i in range(0, half + 1):
    t[i] = mp.power(mp.mpf(i) / mp.mpf(half), r)

# Second half: shifted mesh on [1,2]
for i in range(half + 1, N + 1):
    t[i] = mp.mpf('1') + t[i - half]

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# =========================
# Meshgrid
# =========================
Xplot, Yplot = np.meshgrid(X, t)

Xplot = Xplot.astype(float)
Yplot = Yplot.astype(float)

# =========================
# Convert mpmath → NumPy (SAFE)
# U is (space, time) = (M+3, N+1)
# =========================
V_float = np.array([
    [float(U_error[i, j]) for j in range(N+1)]
    for i in range(M+3)
])

# =========================
# Fix orientation → (time, space)
# =========================
V_float = V_float.T

# =========================
# Safety check (VERY IMPORTANT)
# =========================
print("Xplot shape:", Xplot.shape)
print("V_float shape:", V_float.shape)

if V_float.shape != Xplot.shape:
    raise ValueError("Shape mismatch: check U construction!")

# =========================
# Plot settings
# =========================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120
})

# =========================
# Create figure
# =========================
fig = plt.figure(figsize=(7.5, 5.5))
ax = fig.add_subplot(111, projection='3d')

# =========================
# Surface plot
# =========================
surf = ax.plot_surface(
    Xplot, Yplot, V_float,
    cmap='viridis',
    linewidth=0,
    antialiased=False,
    rstride=1,
    cstride=1
)

surf.set_rasterized(True)

# =========================
# Labels
# =========================
ax.set_xlabel(r"Space $x$", labelpad=10)
ax.set_ylabel(r"Time $t$", labelpad=10)
ax.set_zlabel(r"error plot", labelpad=10)

# =========================
# Scientific Z-axis
# =========================
formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))
ax.zaxis.set_major_formatter(formatter)

# =========================
# View and style
# =========================
ax.view_init(elev=28, azim=225)
ax.set_box_aspect([1, 1, 0.6])
ax.grid(False)

# =========================
# Colorbar (important)
# =========================
cbar = fig.colorbar(surf, shrink=0.6, aspect=12)
cbar.set_label("Solution magnitude")

# =========================
# Save figures
# =========================
plt.tight_layout()
plt.savefig("Numerical_solution.png", dpi=300, bbox_inches="tight")
plt.savefig("Numerical_solution.pdf", bbox_inches="tight")

plt.show()

